In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
RANDOM_STATE = 42

In [3]:
from google.colab import files
uploaded = files.upload()

Saving student-mat.csv to student-mat.csv


This revision fixes preprocessing leakage, aligns hyperparameter tuning with the metric that actually matters(High-rsk recall), puts Sprint 1 and Sprint 2 on the same held out test set, adds cross validation stability check and exposes confidence scores and feature importance.



LOAD DATASET


In [4]:
mat = pd.read_csv('student-mat.csv', sep=';')
mat.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


DEFINE RISK LEVEL- target\n\nUses G1 only(earliest available grade) so the model is trained to predict risk from features that would be known before G1 is even administered.

In [5]:
def risk_level(g1):
    if g1 <= 9:
        return "High"
    elif g1 <= 14:
        return "Medium"
    else:
        return "Low"

mat["Risk_level"] = mat["G1"].apply(risk_level)
mat[["G1", "Risk_level"]].head()

mat.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,Risk_level
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,High
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,High
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,High
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,Low
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,High


SEPARATE FEATURES AND TARGET FIX - explicitly drop G2 and G3 not just G1 so a future edit to the feature lists below vcan never silently reintroduce grade leakage.

In [6]:
X_raw = mat.drop(["G1", "G2", "G3", "Risk_level"], axis=1)
y = mat["Risk_level"]

SPLIT BEFORE PREPROCESSING - split is now happeneing on raw, unencooded data. No encoder or scaler will ever see validation/test rows during fitting.

In [7]:
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X_raw, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print("Training:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Testing:", X_test_raw.shape)

Training: (276, 30)
Validation: (59, 30)
Testing: (60, 30)


ENCODE AND SCALE - fit model only on training data. Key fix: LabelEncoder, OneHoteEncoder, StandardScaler are all fit on X_train_raw only then applied with transform()(not fit_transform) to validation and test. handle_unknown on one hot encoder defensively handles any rare category that lands in val/test but not train.

In [8]:
encoder = LabelEncoder()

binary_features = [
    "school",
    "sex",
    "address",
    "famsize",
    "Pstatus",
    "schoolsup",
    "famsup",
    "paid",
    "activities",
    "nursery",
    "higher",
    "internet",
    "romantic"
]

#  Binary features: LabelEncoder fit on train, applied everywhere
label_encoders = {}
X_train_bin = pd.DataFrame(index=X_train_raw.index)
X_val_bin = pd.DataFrame(index=X_val_raw.index)
X_test_bin = pd.DataFrame(index=X_test_raw.index)

for col in binary_features:
    le = LabelEncoder()
    X_train_bin[col] = le.fit_transform(X_train_raw[col])
    X_val_bin[col] = le.transform(X_val_raw[col])
    X_test_bin[col] = le.transform(X_test_raw[col])
    label_encoders[col] = le



In [9]:
categorical_features = [
    "Mjob",
    "Fjob",
    "reason",
    "guardian"
]

# Categorical features: OneHotEncoder fit on train only
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore")
ohe.fit(X_train_raw[categorical_features])

def encode_categorical(df_raw):
    arr = ohe.transform(df_raw[categorical_features])
    return pd.DataFrame(arr, columns=ohe.get_feature_names_out(categorical_features), index=df_raw.index)

cat_train = encode_categorical(X_train_raw)
cat_val = encode_categorical(X_val_raw)
cat_test = encode_categorical(X_test_raw)


In [10]:
numeric_features = [
    "age",
    "Medu",
    "Fedu",
    "traveltime",
    "studytime",
    "failures",
    "famrel",
    "freetime",
    "goout",
    "Dalc",
    "Walc",
    "health",
    "absences"
]

# Numeric features: StandardScaler fit on train only
scaler = StandardScaler()
scaler.fit(X_train_raw[numeric_features])

def scale_numeric(df_raw):
    arr = scaler.transform(df_raw[numeric_features])
    return pd.DataFrame(arr, columns=numeric_features, index=df_raw.index)

num_train = scale_numeric(X_train_raw)
num_val = scale_numeric(X_val_raw)
num_test = scale_numeric(X_test_raw)



In [11]:
# Combine
X_train_processed = pd.concat([X_train_bin, cat_train, num_train], axis=1)
X_val_processed = pd.concat([X_val_bin, cat_val, num_val], axis=1)
X_test_processed = pd.concat([X_test_bin, cat_test, num_test], axis=1)

X_train_processed.head()

,school,sex,address,famsize,Pstatus,schoolsup,famsup,paid,activities,nursery,...,traveltime,studytime,failures,famrel,freetime,goout,Dalc,Walc,health,absences
25,0,0,1,0,1,0,1,1,0,0,...,-0.580754,-1.256757,2.457656,-3.304528,-1.279600,-0.988608,-0.548381,0.547221,1.030595,0.892955
269,0,0,0,0,1,0,1,0,0,1,...,0.863283,-0.096674,-0.447325,0.052710,-0.273419,1.643441,-0.548381,-0.223356,-0.413284,-0.705922
60,0,0,0,0,1,0,1,0,1,1,...,-0.580754,-0.096674,-0.447325,-2.185448,0.732762,0.766091,0.517486,0.547221,0.308655,-0.020689
138,0,1,1,1,1,0,0,0,0,1,...,-0.580754,-0.096674,1.005166,0.052710,0.732762,0.766091,-0.548381,0.547221,1.030595,-0.705922
43,0,1,1,0,1,1,1,0,0,1,...,-0.580754,-1.256757,-0.447325,1.171790,0.732762,-1.865957,-0.548381,-0.993933,-1.857163,-0.705922


TUNE THE RANDOM FOREST - scoring = accuracy is been replaced with scoring = f1_macro which weights all three classes equally instead of letting the search win by favoring the large "Medium class". class_weight is added to the estimator itself as a second independent lever against class imbalance.

In [12]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"),
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1
)

grid_search.fit(X_train_processed, y_train)
tuned_rf = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)
print("Best CV F1 (macro):", grid_search.best_score_)

Best Parameters: {'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
Best CV F1 (macro): 0.5075991859408591


SPRINT 1 BASELINE - Needed so the two models can be compared fairly in the next cell.

In [13]:
baseline_rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
baseline_rf.fit(X_train_processed, y_train)

RandomForestClassifier(random_state=42)

EVALUATE BOTH MODELS ON THE IDENTICAL UNTOUCHED TEST SET - Both rows are now scored on the same held-out students so any difference reflects tuning, not which students landed in which split. High-risk recall is broken out explicitly that is the number to headline not weighted accuracy.

In [14]:
def evaluate(model, X, y, name):
    preds = model.predict(X)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y, preds),
        "Precision (weighted)": precision_score(y, preds, average="weighted"),
        "Recall (weighted)": recall_score(y, preds, average="weighted"),
        "F1 (weighted)": f1_score(y, preds, average="weighted"),
        "High-risk recall": recall_score(y, preds, labels=["High"], average="micro"),
    }

results = pd.DataFrame([
    evaluate(baseline_rf, X_test_processed, y_test, "Sprint 1-style RF (default)"),
    evaluate(tuned_rf, X_test_processed, y_test, "Sprint 2 tuned RF"),
])
results

,Model,Accuracy,Precision (weighted),Recall (weighted),F1 (weighted),High-risk recall
0,Sprint 1-style RF (default),0.683333,0.751167,0.683333,0.647238,0.636364
1,Sprint 2 tuned RF,0.650000,0.661887,0.650000,0.655090,0.727273


FULL CLASSIFICATION REPORT FOR THE MODEL WE PLAN TO SHIP

In [15]:
tuned_preds = tuned_rf.predict(X_test_processed)
print(classification_report(y_test, tuned_preds))
print(confusion_matrix(y_test, tuned_preds, labels=["High", "Medium", "Low"]))

              precision    recall  f1-score   support

        High       0.76      0.73      0.74        22
         Low       0.36      0.44      0.40         9
      Medium       0.68      0.66      0.67        29

    accuracy                           0.65        60
   macro avg       0.60      0.61      0.60        60
weighted avg       0.66      0.65      0.66        60

[[16  4  2]
 [ 5 19  5]
 [ 0  5  4]]


STABILITY CHECK ACROSS MULTIPLE SPLITS - this reports mean & std macro-f1 across 5 stratified folds of the training data for both models.

In [16]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for name, model in [("Baseline RF", baseline_rf), ("Tuned RF", tuned_rf)]:
    scores = cross_val_score(model, X_train_processed, y_train, cv=cv, scoring="f1_macro")
    print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f} (macro F1 across 5 folds)")

Baseline RF: 0.401 +/- 0.053 (macro F1 across 5 folds)
Tuned RF: 0.456 +/- 0.023 (macro F1 across 5 folds)


CONFIDENCE SCORES - closes the gap between the architecture diagram (which promised prediction confidence) and the code(which never produced one)

In [17]:
probs = tuned_rf.predict_proba(X_test_processed)
confidence_df = pd.DataFrame(probs, columns=tuned_rf.classes_, index=X_test_processed.index)
confidence_df["Predicted"] = tuned_rf.predict(X_test_processed)
confidence_df["Confidence"] = confidence_df[list(tuned_rf.classes_)].max(axis=1)
confidence_df.head()

,High,Low,Medium,Predicted,Confidence
255,0.423560,0.170829,0.405611,High,0.423560
392,0.613532,0.149453,0.237015,High,0.613532
217,0.438408,0.240653,0.320939,High,0.438408
144,0.629383,0.113233,0.257384,High,0.629383
333,0.444649,0.216321,0.339030,High,0.444649


FEATURE IMPORTANCE

In [18]:
importances = pd.Series(
    tuned_rf.feature_importances_, index=X_train_processed.columns
).sort_values(ascending=False)

importances.head(10)

,0
failures,0.072469
freetime,0.059596
studytime,0.054339
absences,0.052292
Walc,0.046999
Dalc,0.045683
health,0.045274
Medu,0.041983
Mjob_other,0.041905
age,0.041773


SAVE MODEL AND FITTED PREPROCESSORS - The original only saved the classifier. Any real deployment needs the exact fitted encoders/ scaler to preprocess a new student's raw answers the same way training data was preprocessed.

In [19]:
joblib.dump({
    "model": tuned_rf,
    "label_encoders": label_encoders,
    "onehot_encoder": ohe,
    "scaler": scaler,
    "binary_features": binary_features,
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
}, "student_risk_model.pkl")

print("Model and preprocessors saved successfully!")

Model and preprocessors saved successfully!


In [21]:
print("Split integrity ")
print("Train+Val+Test =", len(X_train_raw)+len(X_val_raw)+len(X_test_raw), "vs total", len(X_raw))
print("No overlap:", set(X_train_raw.index) & set(X_val_raw.index) & set(X_test_raw.index) == set())

print("\n Scaler was fit on train only ")
print("Scaler saw n_samples_seen_ =", scaler.n_samples_seen_, "(should equal len(X_train_raw), not 395)")

print("\nDid tuning actually help the metric that matters? ")
baseline_high_recall = recall_score(y_test, baseline_rf.predict(X_test_processed), labels=["High"], average="micro")
tuned_high_recall = recall_score(y_test, tuned_rf.predict(X_test_processed), labels=["High"], average="micro")
print(f"Baseline High-risk recall: {baseline_high_recall:.3f}")
print(f"Tuned High-risk recall:    {tuned_high_recall:.3f}")

print("\n Is the model actually predicting all 3 classes, or collapsing to the majority? ")
print(pd.Series(tuned_rf.predict(X_test_processed)).value_counts())

print("\n Does the model trust itself in sensible ways? ")
print(confidence_df["Confidence"].describe())

Split integrity 
Train+Val+Test = 395 vs total 395
No overlap: True

 Scaler was fit on train only 
Scaler saw n_samples_seen_ = 276 (should equal len(X_train_raw), not 395)

Did tuning actually help the metric that matters? 
Baseline High-risk recall: 0.636
Tuned High-risk recall:    0.727

 Is the model actually predicting all 3 classes, or collapsing to the majority? 
Medium    28
High      21
Low       11
Name: count, dtype: int64

 Does the model trust itself in sensible ways? 
count    60.000000
mean      0.434253
std       0.065244
min       0.334296
25%       0.385233
50%       0.422671
75%       0.468789
max       0.629383
Name: Confidence, dtype: float64
